# PathSnap: Multi-Layer Perceptron Training & Evaluation

Train and evaluate the selected MLP classifier using the shared imputed and scaled feature matrices. The workflow mirrors `src/train_models.py`, including deterministic negative sampling and trip-grouped cross-validation.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import confusion_matrix, f1_score, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
MODELS_DIR = ARTIFACTS_DIR / 'models'
PLOTS_DIR = ARTIFACTS_DIR / 'plots'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'dist_orthogonal', 'proj_ratio', 'dist_to_start', 'dist_to_end',
    'distance_rank', 'road_crowding_25_50m', 'arc_length',
    'distance_travelled', 'heading_diff', 'heading_cos', 'speed',
    'time_gap', 'is_time_gap'
]
TARGET_COL = 'is_true_arc'
print('Project root:', PROJECT_ROOT)

## 1. Load the Shared Feature Matrices

The matrices are imputed and scaled by `src/preprocessor.py`, using training trips only. If Parquet support is unavailable, the notebook reads the compressed CSV fallback.


In [ ]:
train_parquet = ARTIFACTS_DIR / 'train_features.parquet'
test_parquet = ARTIFACTS_DIR / 'test_features.parquet'
train_csv = ARTIFACTS_DIR / 'train_features.csv.gz'
test_csv = ARTIFACTS_DIR / 'test_features.csv.gz'

try:
    if not (train_parquet.exists() and test_parquet.exists()):
        raise FileNotFoundError('Parquet feature matrices are missing.')
    train_df = pd.read_parquet(train_parquet)
    test_df = pd.read_parquet(test_parquet)
except (ImportError, FileNotFoundError):
    if not (train_csv.exists() and test_csv.exists()):
        raise FileNotFoundError('Run src/preprocessor.py to generate the feature matrices.')
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)

required = set(FEATURE_COLS + [TARGET_COL])
if required - set(train_df.columns) or required - set(test_df.columns):
    raise ValueError('Feature matrices do not match the current feature schema.')
print(f'Train: {len(train_df):,} candidates from {train_df.trip_id.nunique()} trips')
print(f'Test:  {len(test_df):,} candidates from {test_df.trip_id.nunique()} trips')

## 2. Configure the MLP and Training Sample

When there are more than 300,000 training rows, retain all positive candidates and sample up to five negatives per positive with seed 42, as in the benchmark script.


In [ ]:
if len(train_df) > 300_000:
    positive = train_df[train_df[TARGET_COL] == 1]
    negative = train_df[train_df[TARGET_COL] == 0].sample(
        n=min(len(train_df) - len(positive), len(positive) * 5),
        random_state=42
    )
    train_subset = pd.concat([positive, negative], ignore_index=True)
else:
    train_subset = train_df.copy()

X_train = train_subset[FEATURE_COLS].to_numpy(dtype=float)
y_train = train_subset[TARGET_COL].to_numpy(dtype=int)
groups_train = train_subset['trip_id'].to_numpy()
X_test = test_df[FEATURE_COLS].to_numpy(dtype=float)
y_test = test_df[TARGET_COL].to_numpy(dtype=int)

model = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
    alpha=0.0001, batch_size=512, learning_rate_init=0.001,
    max_iter=150, early_stopping=True, n_iter_no_change=10,
    random_state=42
)
print('Training rows:', len(train_subset), '| positive fraction:', round(float(y_train.mean()), 4))
print(model)

## 3. Group-Aware Cross-Validation

The 3 folds split by trip ID, preventing candidates from the same trip from appearing in both training and validation. Top-1 accuracy measures whether the highest-scoring candidate per GPS observation is correct.


In [ ]:
def observation_top1_accuracy(frame, probabilities):
    scored = frame[['trip_id', 'obs_idx', TARGET_COL]].copy()
    scored['probability'] = probabilities
    best = scored.loc[scored.groupby(['trip_id', 'obs_idx'])['probability'].idxmax()]
    return float(best[TARGET_COL].mean())

cv = GroupKFold(n_splits=3)
cv_rows = []
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train, groups_train), start=1):
    fold_model = clone(model)
    fold_model.fit(X_train[train_idx], y_train[train_idx])
    valid_prob = fold_model.predict_proba(X_train[valid_idx])[:, 1]
    valid_pred = (valid_prob >= 0.5).astype(int)
    valid_frame = train_subset.iloc[valid_idx]
    row = {
        'fold': fold,
        'macro_f1': f1_score(y_train[valid_idx], valid_pred, average='macro', zero_division=0),
        'top1_accuracy': observation_top1_accuracy(valid_frame, valid_prob),
        'log_loss': log_loss(y_train[valid_idx], valid_prob, labels=[0, 1])
    }
    cv_rows.append(row)
    print(f"Fold {fold}: macro F1={row['macro_f1']:.4f}, top-1={row['top1_accuracy']:.4f}, log loss={row['log_loss']:.4f}")

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
print('Mean CV macro F1:', round(float(cv_results['macro_f1'].mean()), 4))
print('Mean CV top-1 accuracy:', round(float(cv_results['top1_accuracy'].mean()), 4))

## 4. Train on All Training Trips and Evaluate Student Test


In [ ]:
model.fit(X_train, y_train)
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
test_results = {
    'macro_f1': f1_score(y_test, test_pred, average='macro', zero_division=0),
    'top1_accuracy': observation_top1_accuracy(test_df, test_prob),
    'log_loss': log_loss(y_test, test_prob, labels=[0, 1])
}
display(pd.Series(test_results, name='student_test'))

model_path = MODELS_DIR / 'multi-layer_perceptron.joblib'
joblib.dump(model, model_path)
print('Saved model:', model_path)
print('Training epochs:', model.n_iter_, '| final loss:', round(float(model.loss_), 5))

## 5. Training Curve and Test Diagnostics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(model.loss_curve_) + 1), model.loss_curve_, color='#d95f02')
axes[0].set(title='MLP Training Loss', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[0].grid(alpha=0.25)

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-arc', 'True arc'], yticklabels=['Non-arc', 'True arc'], ax=axes[1])
axes[1].set(title='MLP Student-Test Confusion Matrix', xlabel='Predicted', ylabel='Actual')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'mlp_training_and_confusion.png', dpi=200)
plt.show()